# Stacking Pipeline: SINAN + Recife

This notebook trains a stacking model using:

Base learners:
- `CatBoostClassifier`
- `RandomForestClassifier`
- `HistGradientBoostingClassifier`
- `LogisticRegression`

Meta learner:
- `MLPClassifier`

Input files:
- `sinan_fin.csv`
- `recife_fin.csv`

The two datasets are loaded separately, aligned by common columns, concatenated, preprocessed, and trained in one stacking pipeline.

In [ ]:
import os
os.listdir('/kaggle/input/datasets/kaleykim297/dengue')
os.chdir('/kaggle/input/datasets/kaleykim297/dengue')

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import json
import joblib
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    balanced_accuracy_score,
    f1_score,
    accuracy_score,
)

from sklearn.ensemble import (
    RandomForestClassifier,
    HistGradientBoostingClassifier,
    StackingClassifier,
)
from sklearn.linear_model import RidgeClassifier, LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.neural_network import MLPClassifier

try:
    from catboost import CatBoostClassifier
except ImportError as e:
    raise ImportError("CatBoost is not installed. Run: pip install catboost") from e

RANDOM_STATE = 42

## 1. Load datasets separately

Change the file paths if your files are in a different folder.

In [ ]:
SINAN_PATH = Path("sinan_fin.csv")
RECIFE_PATH = Path("recife_fin.csv")

sinan = pd.read_csv(SINAN_PATH, low_memory=False)
recife = pd.read_csv(RECIFE_PATH, low_memory=False)

print("SINAN shape:", sinan.shape)
print("Recife shape:", recife.shape)

display(sinan.head())
display(recife.head())

## 2. Align and concatenate

This uses the intersection of columns. That is stricter than union-based merging, but safer for one common model.

In [ ]:
sinan["source"] = "sinan"
recife["source"] = "recife"

common_cols = sorted(set(sinan.columns) & set(recife.columns))

sinan_common = sinan[common_cols].copy()
recife_common = recife[common_cols].copy()

df = pd.concat([sinan_common, recife_common], axis=0, ignore_index=True)

print("Common columns:", len(common_cols))
print("Merged shape:", df.shape)
print(df["source"].value_counts())

display(df.head())

## 3. Choose target column

In [ ]:
print(df["target"].value_counts(dropna=False))

## 4. Clean target and remove leakage columns

Missing targets are dropped. Target-like columns are not allowed as input features.

In [ ]:
df = df.drop(columns=["target_with_severity", "Unnamed: 0"])
df = df.dropna(subset=["target"]).copy()
df["target"] = df["target"].astype(str)

X = df.drop(columns=["target"])
y = df["target"].copy()

le = LabelEncoder()
y = le.fit_transform(y)

print("X shape:", X.shape)
print("Classes:")
for i, cls in enumerate(le.classes_):
    print(i, "->", cls)

## 5. Detect numeric and categorical columns

The same preprocessing is used for every base learner because `StackingClassifier` expects a common input representation.

In [ ]:
X = X.replace([np.inf, -np.inf], np.nan)

num_cols = X.select_dtypes(include=["number", "bool"]).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]

print("Numeric columns:", len(num_cols))
print(num_cols)
print("Categorical columns:", len(cat_cols))
print(cat_cols)

## 6. Train/test split

Use stratification because class imbalance is expected.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train distribution:")
print(pd.Series(y_train).value_counts().sort_index())
print("Test distribution:")
print(pd.Series(y_test).value_counts().sort_index())

## 7. Preprocessing pipeline

- Numeric columns: median imputation + scaling
- Categorical columns: most-frequent imputation + one-hot encoding

`OneHotEncoder` outputs dense arrays because `HistGradientBoostingClassifier` is safer with dense input.

In [ ]:
try:
    onehot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    onehot = OneHotEncoder(handle_unknown="ignore", sparse=False)

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", onehot),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, num_cols),
        ("cat", categorical_pipeline, cat_cols),
    ],
    remainder="drop",
)

## 8. Define base learners

The base learners are deliberately diverse:

- CatBoost: gradient boosting
- RandomForest: bagging/tree ensemble
- HistGradientBoosting: sklearn boosting
- LogisticRegression: linear baseline

In [ ]:
cb_best_params = {'iterations': 500, 
                  'learning_rate': 0.08995147228189702, 
                  'depth': 8, 
                  'l2_leaf_reg': 4.2403143657545375, 
                  'random_strength': 0.07495849797769727, 
                  'border_count': 182}
catboost_base = CatBoostClassifier(
    loss_function="MultiClass",
    eval_metric="TotalF1",
    random_seed=RANDOM_STATE,
    verbose=0,
    auto_class_weights="Balanced",
    train_dir="/kaggle/working/catboost_info",
    **cb_best_params
)

rf_best_params = {'n_estimators': 1000, 
                 'max_depth': 17, 
                 'min_samples_split': 16, 
                 'min_samples_leaf': 2,
                 'max_features': 'sqrt',
                 'class_weight': 'balanced_subsample',
                 'criterion': 'entropy'}
rf_base = RandomForestClassifier(
    random_state=RANDOM_STATE,
    n_jobs=-1,
    **rf_best_params,
    
)

hgb_best_params = {'learning_rate': 0.03961867790406585,
                   'max_iter': 780,
                   'max_depth': 6,
                   'max_leaf_nodes': 28,
                   'min_samples_leaf': 18,
                   'l2_regularization': 0.0001893704954163131,
                   'max_bins': 177,
                   'validation_fraction': 0.1407023547660844,
                   'n_iter_no_change': 27}
hgb_base = HistGradientBoostingClassifier(
    random_state=RANDOM_STATE,
    **hgb_best_params,
)

rc_best_params = {'max_iter': 1480,
                  'alpha': 17.971050956389703,
                  'fit_intercept': False,
                  'tol': 0.009933336502723955}
ridge_base = CalibratedClassifierCV(
    estimator=RidgeClassifier(
        class_weight="balanced",
        solver="saga",
        random_state=RANDOM_STATE,
        **rc_best_params,
    ),
    method="sigmoid",
    cv=3
)

lr_best_params = {'C': 0.6924150974751488,
                  'class_weight': 'balanced',
                  'tol': 0.0024605046266131365,
                  'intercept_scaling': 0.17173755394370507}
lr_base = LogisticRegression(
    random_state=RANDOM_STATE,
    max_iter=4000,
    solver="liblinear",
    l1_ratio=0,
    dual=True,
    fit_intercept=True,
    **lr_best_params,
)

base_estimators = [
    ("cb", catboost_base),
    ("rf", rf_base),
    ("hgb", hgb_base),
    ("rc", ridge_base),
]

## 9. Define MLP meta learner

The MLP receives out-of-fold predicted probabilities from the base learners.

Keep it small first. A large MLP here is overkill and can overfit.

In [ ]:
meta_mlp = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("mlp", MLPClassifier(
        hidden_layer_sizes=(16,),
        activation="tanh",
        solver="lbfgs",
        alpha=1e-2,
        max_iter=1000,
        random_state=RANDOM_STATE,
    ))
])

## 10. Build stacking classifier

`stack_method="predict_proba"` passes class probabilities to the MLP meta learner.

Start with `passthrough=False`. If results are weak, test `passthrough=True` later.

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

stack_model = StackingClassifier(
    estimators=base_estimators,
    final_estimator=meta_mlp,
    stack_method="predict_proba",
    cv=cv,
    passthrough=False,
    n_jobs=1,
    verbose=1,
)

stacking_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("stack", stack_model),
])

stacking_pipeline

## 11. Train stacking model

This can take time because stacking trains base learners inside cross-validation.

In [ ]:
X_debug = preprocessor.fit_transform(X_train)

print("Any NaN:", np.isnan(X_debug).any())
print("Any inf:", np.isinf(X_debug).any())
print("Max abs value:", np.nanmax(np.abs(X_debug)))
print("Shape:", X_debug.shape)

In [ ]:
stacking_pipeline.fit(X_train, y_train)
print("Training complete.")

## 12. Evaluate stacking model

In [ ]:
y_pred = stacking_pipeline.predict(X_test)

metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
    "macro_f1": f1_score(y_test, y_pred, average="macro"),
    "weighted_f1": f1_score(y_test, y_pred, average="weighted"),
}

print(metrics)
print("Classification report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(cm, index=le.classes_, columns=le.classes_)
display(cm_df)

## 13. Probability review

Use this for confidence-thresholding later.

In [ ]:
y_proba = stacking_pipeline.predict_proba(X_test)

proba_df = pd.DataFrame(
    y_proba,
    columns=[f"proba_{cls}" for cls in le.classes_],
)

proba_review = pd.concat([
    pd.DataFrame({
        "true_label": le.inverse_transform(y_test),
        "pred_label": le.inverse_transform(y_pred),
        "max_proba": y_proba.max(axis=1),
    }).reset_index(drop=True),
    proba_df.reset_index(drop=True),
], axis=1)

display(proba_review.head(20))

## 14. Compare individual base learners

Do not assume stacking is better. Check it.

In [ ]:
base_results = []

for name, model in base_estimators:
    print(f"Training {name}...")
    pipe = Pipeline(steps=[
        ("preprocess", preprocessor),
        ("model", model),
    ])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    base_results.append({
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, pred),
        "macro_f1": f1_score(y_test, pred, average="macro"),
        "weighted_f1": f1_score(y_test, pred, average="weighted"),
    })

base_results_df = pd.DataFrame(base_results).sort_values("macro_f1", ascending=False)
display(base_results_df)

## 15. Save artifacts

This saves:

- fitted stacking pipeline
- label encoder
- feature columns
- class names

In [ ]:
ARTIFACT_DIR = Path("artifacts_stacking_mlp")
ARTIFACT_DIR.mkdir(exist_ok=True)

joblib.dump(stacking_pipeline, ARTIFACT_DIR / "stacking_mlp_pipeline.pkl")
joblib.dump(le, ARTIFACT_DIR / "label_encoder.pkl")

with open(ARTIFACT_DIR / "feature_columns.json", "w", encoding="utf-8") as f:
    json.dump(feature_cols, f, indent=2)

with open(ARTIFACT_DIR / "class_names.json", "w", encoding="utf-8") as f:
    json.dump(le.classes_.tolist(), f, indent=2)

print("Saved artifacts to:", ARTIFACT_DIR)

## 16. Inference example

In [ ]:
sample = X_test.iloc[:5].copy()

loaded_pipeline = joblib.load(ARTIFACT_DIR / "stacking_mlp_pipeline.pkl")
loaded_le = joblib.load(ARTIFACT_DIR / "label_encoder.pkl")

sample_pred = loaded_pipeline.predict(sample)
sample_labels = loaded_le.inverse_transform(sample_pred)
print(sample_labels)

sample_proba = loaded_pipeline.predict_proba(sample)
display(pd.DataFrame(sample_proba, columns=loaded_le.classes_))